In [1]:
# import necessary libraries
import pandas as pd
import numpy as np
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import OneHotEncoder
from sklearn.preprocessing import OrdinalEncoder
from sklearn.model_selection import train_test_split
from sklearn.compose import ColumnTransformer

In [2]:
# df = pd.read_csv(
#     r"C:\Users\infor\OneDrive\Desktop\Machine Learning\DataSets\covid_toy.csv"
# )
df = pd.read_csv("../DataSets/covid_toy.csv")

In [3]:
df.head()

,age,gender,fever,cough,city,has_covid
0,60,Male,103.0,Mild,Kolkata,No
1,27,Male,100.0,Mild,Delhi,Yes
2,42,Male,101.0,Mild,Delhi,No
3,31,Female,98.0,Mild,Kolkata,No
4,65,Female,101.0,Mild,Mumbai,No


In [4]:
# Checking for null values
df.isnull().sum()

age           0
gender        0
fever        10
cough         0
city          0
has_covid     0
dtype: int64

In [5]:
df["cough"].value_counts()

cough
Mild      62
Strong    38
Name: count, dtype: int64

In [6]:
X = df.drop(columns="has_covid")

In [7]:
y = df["has_covid"]

In [8]:
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42
)

In [9]:
X_train

,age,gender,fever,cough,city
55,81,Female,101.0,Mild,Mumbai
88,5,Female,100.0,Mild,Kolkata
26,19,Female,100.0,Mild,Kolkata
42,27,Male,100.0,Mild,Delhi
69,73,Female,103.0,Mild,Delhi
...,...,...,...,...,...
60,24,Female,102.0,Strong,Bangalore
71,75,Female,104.0,Strong,Delhi
14,51,Male,104.0,Mild,Bangalore
92,82,Female,102.0,Strong,Kolkata


In [ ]:
si = SimpleImputer()
X_train_fever = si.fit_transform(X_train[["fever"]])
X_test_fever = si.transform(X_test[["fever"]])

X_train_fever.shape

(80, 1)

OrdinalEncoding -> cough


In [ ]:
oe = OrdinalEncoder(categories=[["Mild", "Strong"]])
X_train_cough = oe.fit_transform(X_train[["cough"]])

X_test_cough = oe.transform(X_test[["cough"]])
X_train_cough.shape

(80, 1)

OneHotEncoding -> gender, city


In [ ]:
ohe = OneHotEncoder(drop="first", sparse_output=False)
X_train_gender_city = ohe.fit_transform(X_train[["gender", "city"]])

X_test_gender_city = ohe.transform(X_test[["gender", "city"]])

X_train_gender_city.shape

(80, 4)

In [13]:
# Extracting Age
X_train_age = X_train.drop(columns=["gender", "fever", "cough", "city"]).values

# aslo the test data
X_test_age = X_test.drop(columns=["gender", "fever", "cough", "city"]).values

X_train_age.shape

(80, 1)

In [14]:
X_train_transformed = np.concatenate(
    (X_train_age, X_train_fever, X_train_gender_city, X_train_cough), axis=1
)
# also the test data
X_test_transformed = np.concatenate(
    (X_test_age, X_test_fever, X_test_gender_city, X_test_cough), axis=1
)

X_train_transformed.shape

(80, 7)

In [15]:
X_test_transformed.shape

(20, 7)

Using ColumnTransformer


In [16]:
transformer = ColumnTransformer(
    transformers=[
        ("si", SimpleImputer(), ["fever"]),
        ("oe", OrdinalEncoder(categories=[["Mild", "Strong"]]), ["cough"]),
        ("ohe", OneHotEncoder(sparse_output=False, drop="first"), ["gender", "city"]),
    ],
    remainder="passthrough",
)

In [17]:
transformer.fit_transform(X_train).shape

(80, 7)

In [18]:
transformer.transform(X_test).shape

(20, 7)

In [19]:
new_df = np.concatenate(
    (transformer.fit_transform(X_train), transformer.transform(X_test))
)

In [20]:
col_names = (
    ["fever", "cough"] + list(ohe.get_feature_names_out(["gender", "city"])) + ["age"]
)
new_df = pd.DataFrame(new_df, columns=col_names)
new_df.head()

,fever,cough,gender_Male,city_Delhi,city_Kolkata,city_Mumbai,age
0,101.0,0.0,0.0,0.0,0.0,1.0,81.0
1,100.0,0.0,0.0,0.0,1.0,0.0,5.0
2,100.0,0.0,0.0,0.0,1.0,0.0,19.0
3,100.0,0.0,1.0,1.0,0.0,0.0,27.0
4,103.0,0.0,0.0,1.0,0.0,0.0,73.0
